# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`
This notebook provides an end-to-end guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, enabling reproducible data loading and programmatic metadata inspection.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL -- FAIR² Croissant schema
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset (metadata and record definitions)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and field `@id`s for orientation within the dataset schema.

In [ ]:
# List available record sets and their fields, referencing `@id`
record_sets = dataset.record_sets  # this is a list of RecordSet objects

print("Available record sets:")
for rs in record_sets:
    print(f"- @id: {rs.id}")
    print(f"  name: {rs.name}")
    print(f"  description: {rs.description}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}")
        print(f"      name: {field.name}")
        if hasattr(field, 'description') and field.description:
            print(f"      description: {field.description}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

All references to record sets and fields are made using their `@id` values as shown above.

In [ ]:
# Extract all record sets into DataFrames
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    # Stream records for this record set by `@id`
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Display the names of record sets loaded
print("Loaded record set DataFrames:")
for rsid, df in dataframes.items():
    print(f"@id: {rsid}  (columns: {list(df.columns)})")

# Preview the main tabular record set (choose the largest one)
main_record_set = max(dataframes.keys(), key=lambda x: len(dataframes[x]))  # heuristic: the biggest one
print(f"\nDisplaying .head() of primary record set @id: {main_record_set}")
display(dataframes[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, and grouping.

In this example, we demonstrate filtering and normalization for an available numeric field. Check the DataFrame columns above to select a numeric field's `@id`.

In [ ]:
# Identify a likely numeric field from the columns (e.g., 'cr:field_age', 'cr:field_years_between_diagnosis', etc.)
# If the column names are not obvious, display the first few rows again:
df = dataframes[main_record_set]
print("Columns available in the primary record set:")
print(df.columns.tolist())
display(df.head())

# For this FAIR² dataset, let's assume there is a numeric field named 'cr:field_years_between_diagnosis'.
# (If your output shows a different field, replace accordingly with the numeric column's @id.)
numeric_field = None
for col in df.columns:
    if "year" in col.lower() or "age" in col.lower() or "interval" in col.lower():
        numeric_field = col
        break

if numeric_field:
    print(f"Proceeding with numeric field: {numeric_field}")
    # Drop rows with missing values in the numeric field
    filtered_df = df.dropna(subset=[numeric_field]).copy()
    # Filter: e.g., keep only entries with years between diagnoses > 1
    try:
        threshold = 1
        filtered_df = filtered_df[filtered_df[numeric_field].astype(float) > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} entries")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
        ) / filtered_df[numeric_field].astype(float).std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Group by a categorical field if present (e.g., 'cr:field_sex')
        group_field = None
        for gcol in df.columns:
            if "sex" in gcol.lower() or "msi" in gcol.lower() or "site" in gcol.lower():
                group_field = gcol
                break
        if group_field:
            print(f"\nGrouped by {group_field}:")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
            display(grouped_df)
    except Exception as e:
        print(f"Numeric filtering/normalization failed: {e}")
else:
    print("No obvious numeric field found for EDA demonstration.")

## 5. Visualization
Visualize distributions and relationships in the data using the available numeric and categorical fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Histogram of the numeric field
if numeric_field:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field].astype(float), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    # If group_field is found, show boxplot by group
    if group_field:
        plt.figure(figsize=(7,4))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion

This notebook demonstrated how to load, inspect, and analyze the FAIR² dataset on second primary colorectal cancer using the Croissant schema and the `mlcroissant` Python library. 

- Data and schema access are fully reproducible and referencable by persistent `@id` values from the Croissant metadata.
- We previewed record sets and fields, extracted the main tabular data into a DataFrame, and performed light exploratory data analysis (filtering, normalization, and group comparison).
- Visualization of numeric fields by pathology or demographic category is possible using the schema.

Continue to explore using the identified field and record set `@id`s to perform domain-specific analysis, modeling, or downstream machine learning as needed!